In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

In [2]:
df = pd.read_csv('cleaned_bike_data.csv')
df = df[df['trip_duration'] <= 60].copy()
print(df.shape)


(216610, 18)


In [3]:
le_target = LabelEncoder()
le_ride = LabelEncoder()
le_day = LabelEncoder()

df['target'] = le_target.fit_transform(df['user_type'])
df['ride_type_enc'] = le_ride.fit_transform(df['ride_type'])
df['day_of_week_enc'] = le_day.fit_transform(df['day_of_week'])

print(df[['user_type', 'target']].drop_duplicates())


  user_type  target
0    member       1
8    casual       0


In [4]:
features = ['ride_type_enc', 'trip_duration', 'hour', 'day_of_week_enc', 'month', 'distance_km']

X = df[features]
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train:", X_train.shape)
print("Test:", X_test.shape)


Train: (173288, 6)
Test: (43322, 6)


In [5]:
rf_model = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

print(classification_report(y_test, rf_pred))


              precision    recall  f1-score   support

           0       0.70      0.30      0.42     12241
           1       0.77      0.95      0.85     31081

    accuracy                           0.77     43322
   macro avg       0.74      0.62      0.64     43322
weighted avg       0.75      0.77      0.73     43322



In [6]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(n_estimators=200, max_depth=8, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)
print(classification_report(y_test, xgb_pred))

              precision    recall  f1-score   support

           0       0.69      0.33      0.45     12241
           1       0.78      0.94      0.85     31081

    accuracy                           0.77     43322
   macro avg       0.73      0.64      0.65     43322
weighted avg       0.75      0.77      0.74     43322



In [7]:
auc = roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:,1])
print(f"AUC-ROC: {auc:.4f}")


AUC-ROC: 0.7509
